In [1]:
import jiwer
from datasets import load_dataset, Audio

N_SAMPLES = 10

fleurs_id = load_dataset("google/fleurs", "id_id", split="validation")  # cached on disk after first download, no streaming refetch
fleurs_id = fleurs_id.cast_column("audio", Audio(decode=False))  # avoid torchcodec/torch dep, decode via soundfile instead
fleurs_id_samples = list(fleurs_id.select(range(N_SAMPLES)))

print(f"loaded {len(fleurs_id_samples)} samples from fleurs id_id validation split")

parquet-data/id_id/train-00000-of-00001.(…):   0%|          | 0.00/2.09G [00:00<?, ?B/s]

parquet-data/id_id/validation-00000-of-0(…):   0%|          | 0.00/266M [00:00<?, ?B/s]

parquet-data/id_id/test-00000-of-00001.p(…):   0%|          | 0.00/543M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2579 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/350 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]

loaded 10 samples from fleurs id_id validation split


In [2]:
from dataclasses import dataclass
from faster_whisper.transcribe import Segment

@dataclass
class TranscriptionResult:
    text: str
    language: str
    language_probability: float
    segments: list[Segment]
    audio_duration_s: float
    inference_time_s: float

    @property
    def realtime_factor(self) -> float:
        return self.audio_duration_s / self.inference_time_s

In [3]:
from faster_whisper import WhisperModel

device = "cpu"
compute_type = "float32"

# Audio config
SAMPLE_RATE = 16_000
CHANNELS = 1
DTYPE = "float32"

# Chunking config
CHUNK_DURATION_S = 3       # transcribe every N seconds
OVERLAP_DURATION_S = 0.5   # overlap between chunks to avoid cutting words mid-sentence
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)

In [4]:
import time

def transcribe(model: WhisperModel, audio_path: str, language: str | None = None, beam_size: int = 5) -> TranscriptionResult:
    """
    Transcribe an audio file.

    Args:
        model: Loaded WhisperModel instance.
        audio_path: Path to audio file (wav, mp3, m4a, etc.).
        language: ISO 639-1 code e.g. "id" for Indonesian, "en" for English.
                  None = auto-detect.
        beam_size: Beam search width. Higher = more accurate, slower.

    Returns:
        TranscriptionResult with text, segments, timing info.
    """

    t0 = time.perf_counter()
    
    segments_gen, info = model.transcribe(
        audio_path,
        language=language,
        beam_size=beam_size,
        vad_filter=True,
        vad_parameters=dict(
            min_silence_duration_ms=2000,  # wait 2s of silence before cutting
            speech_pad_ms=400,             # pad speech edges to avoid clipping
            threshold=0.3,                 # lower = more sensitive to speech
        ),
        word_timestamps=True
    )

    segments = list(segments_gen)
    full_text = " ".join(seg.text.strip() for seg in segments)
    elapsed = time.perf_counter() - t0

    return TranscriptionResult(
        text=full_text,
        language=info.language,
        language_probability=info.language_probability,
        segments=segments,
        audio_duration_s=info.duration,
        inference_time_s=elapsed
    )

In [5]:
import time
import threading
import pyaudio
from RealtimeSTT import AudioToTextRecorder

def find_input_device_index(name_substring: str) -> int:
    """
    Look up a PyAudio input device index by (partial) name.

    Args:
        name_substring: case-insensitive substring to match, e.g. "MacBook Pro Microphone".

    Returns:
        Device index to pass as input_device_index.
    """
    p = pyaudio.PyAudio()
    try:
        for i in range(p.get_device_count()):
            info = p.get_device_info_by_index(i)
            if info["maxInputChannels"] > 0 and name_substring.lower() in info["name"].lower():
                return i
    finally:
        p.terminate()
    raise ValueError(f"no input device matching {name_substring!r}")

MACBOOK_MIC_INDEX = find_input_device_index("MacBook Pro Microphone")

def make_recorder(
    language: str | None = None,
    model_size: str = "large-v3-turbo",
    on_realtime_update=None,
    on_stabilized=None,
    input_device_index: int | None = MACBOOK_MIC_INDEX,
) -> AudioToTextRecorder:
    """
    Build a RealtimeSTT recorder backed by faster-whisper.

    RealtimeSTT owns mic capture (PyAudio) + VAD (silero/webrtc) + chunking
    internally, so no manual buffering/overlap logic is needed here.

    Args:
        language: ISO 639-1 code e.g. "id". None = auto-detect.
        model_size: faster-whisper model size/name for the *final* transcription.
        on_realtime_update: callback(text) fired continuously with the live,
                             not-yet-finalized partial transcript.
        on_stabilized: callback(text) fired when the realtime partial stops changing.
        input_device_index: PyAudio device index to force capture from a specific mic
                             (defaults to the MacBook Pro's built-in microphone,
                             bypassing whatever the OS default input device is).

    Returns:
        Configured AudioToTextRecorder (call .text() in a loop, or .shutdown() to stop).
    """
    return AudioToTextRecorder(
        model=model_size,
        language=language or "",
        device="cpu",
        compute_type=compute_type,
        sample_rate=SAMPLE_RATE,
        input_device_index=input_device_index,
        enable_realtime_transcription=on_realtime_update is not None,
        realtime_model_type=model_size,
        on_realtime_transcription_update=on_realtime_update,
        on_realtime_transcription_stabilized=on_stabilized,
        spinner=False,
    )

def stream_transcribe(
    stop_event: threading.Event,
    language: str | None = None,
    output=None,
    debug_log: list | None = None,
):
    def emit(msg: str):
        if output is not None:
            with output:
                print(msg)
        else:
            print(msg)

    start_time = time.time()
    utterance_idx = 0

    def on_final(text: str):
        nonlocal utterance_idx
        utterance_idx += 1
        elapsed = time.time() - start_time
        emit(f"[utterance {utterance_idx:03d} | {elapsed:6.1f}s] {text}")
        if debug_log is not None:
            debug_log.append(dict(idx=utterance_idx, elapsed_s=elapsed, text=text))

    recorder = make_recorder(language=language)

    def transcription_loop():
        while not stop_event.is_set():
            recorder.text(on_final)

    t = threading.Thread(target=transcription_loop, daemon=True)
    t.start()

    try:
        while not stop_event.is_set():
            time.sleep(0.1)
    finally:
        recorder.shutdown()
        t.join(timeout=5)

In [6]:
debug_log = []
stop_event = threading.Event()
try:
    stream_transcribe(stop_event=stop_event, language="id", debug_log=debug_log)
except KeyboardInterrupt:
    stop_event.set()

[utterance 001 |   19.0s] Kecil itu sayuran yang diberi sambal kaca. Rasanya pedas, manis, dan gurih.
[utterance 002 |   46.9s] If you want to understand how things work, especially these days, this game is for you. You're not just playing, you're actually learning.
[utterance 003 |   50.5s] Terima kasih.
[utterance 004 |   72.0s] Selamat siang. Meja untuk berapa orang? Untuk dua orang, mbak. Silahkan duduk. Boleh minta menu-nya, mbak? Baik, ini menu. Apa itu pecel, mbak?
[utterance 005 |   79.6s] Nah tapi kalau penuhnya udah banyak gitu, benuh-benuh ini.
RealtimeSTT shutting down
[utterance 006 |   83.0s] 


In [ ]:
# import io

# normalizer = jiwer.Compose([
#     jiwer.ToLowerCase(),
#     jiwer.RemovePunctuation(),
#     jiwer.RemoveMultipleSpaces(),
#     jiwer.Strip(),
#     jiwer.ReduceToListOfListOfWords(),
# ])

# results = []

# for i, sample in enumerate(fleurs_id_samples):
#     audio = sample["audio"]
#     reference = sample["transcription"]

#     audio_np, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")

#     with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as f:
#         sf.write(f.name, audio_np, sr)
#         result = transcribe(model, f.name, language="id")

#     wer = jiwer.wer(reference, result.text, reference_transform=normalizer, hypothesis_transform=normalizer)
#     results.append(dict(
#         idx=i,
#         reference=reference,
#         hypothesis=result.text,
#         wer=wer,
#         rtf=result.realtime_factor,
#     ))
#     print(f"[{i:02d}] WER={wer:.3f} RTF={result.realtime_factor:.2f}")
#     print(f"     ref: {reference}")
#     print(f"     hyp: {result.text}")

# avg_wer = sum(r["wer"] for r in results) / len(results)
# avg_rtf = sum(r["rtf"] for r in results) / len(results)
# print(f"\naverage WER: {avg_wer:.3f} | average RTF: {avg_rtf:.2f}")